In [3]:
import re
from typing import Dict, Any, List
import pprint

import manim as mn
from manim import *

In [4]:
config.verbosity = "WARNING"
config.background_color = BLUE_A
Text.set_default(color=BLACK, stroke_color=BLACK)
MathTex.set_default(color=BLACK, stroke_color=BLACK)

In [5]:
#pokemon name, -form(optional), (gender)(optional), @ item(optional)
pokemon_item_re = re.compile(r'^\s*([A-Za-z0-9\-\s]+?)(?:-(\S+))?(?:\s+\((M|F|N|Genderless)\))?\s*(?:@\s*(.+))?\s*$', re.I)
#groups 1+ letter, :, (.+)rest of line
ability_extras_re = re.compile(r'^\s*([A-Za-z]+):\s*(.+)$', re.I)
#groups of 1+ digit, 0+ whitespace, 1-3 letter stat
evs_re = re.compile(r'(\d+)\s*([A-Za-z]{1,3})')
#groups of whitespace - whitespace, rest of line
moves_re = re.compile(r'^\s*-\s*(.+)$')


def parse_evs(evs_str: str) -> Dict[str, int]:
    evs = {}
    for match in evs_re.finditer(evs_str):
        val, stat = match.groups()
        stat_map = {'HP': 'hp', 'Atk': 'atk', 'Def': 'def', 'SpA': 'spa', 'SpD': 'spd', 'Spe': 'spe',
                    'Spd': 'spd', 'Sp': 'spd'}  # include some variants
        key = stat_map.get(stat, stat.lower())
        evs[key] = int(val)
    return evs


def parse_pokemon_set(text: str) -> Dict[str, Any]:
    """
    Parse a single Pokemon set in common showdown-style format into a dict.
    Returns keys: name, form (optional), gender (optional), item (optional),
    ability (optional), level (int, optional), shiny (bool), evs (dict),
    nature (optional), moves (list).
    """
    lines = [l.rstrip() for l in text.strip().splitlines() if l.strip() != '']
    if not lines:
        return {}

    result: Dict[str, Any] = {
        'name': None,
        'form': None,
        'gender': None,
        'item': None,
        'ability': None,
        'level': None,
        'shiny': False,
        'evs': {},
        'nature': None,
        'moves': []
    }

    # Header line (first non-empty line)
    header = lines[0]
    m = pokemon_item_re.match(header)
    if m:
        name, form, gender, item = m.groups()
        result['name'] = name.strip()
        if form:
            result['form'] = form.strip()
        if gender:
            g = gender.strip()
            result['gender'] = g
        if item:
            result['item'] = item.strip()

    # parse remaining lines
    for line in lines[1:]:
        # moves
        moves = moves_re.match(line)
        if moves:
            move = moves.group(1).strip()
            result['moves'].append(move)
            continue

        # key: value lines
        extras = ability_extras_re.match(line)
        if extras:
            key, val = extras.groups()
            key_lower = key.lower()
            if key_lower == 'ability':
                result['ability'] = val.strip()
            elif key_lower == 'level':
                try:
                    result['level'] = int(val.strip())
                except ValueError:
                    result['level'] = 50
            elif key_lower == 'shiny':
                result['shiny'] = True
            elif key_lower == 'evs':
                result['evs'] = parse_evs(val)
            else:
                # fallback: store unknown key
                result[key_lower] = val.strip()
            continue

        # nature line (ends with "Nature")
        if line.strip().endswith('Nature'):
            # example "Jolly Nature"
            parts = line.strip().split()
            if parts:
                result['nature'] = parts[0].strip()
            continue


    return result

def parse_pokemon_team(text: str) -> list[dict[str, Any]]:
    sets = text.strip().split('\n\n')
    return [parse_pokemon_set(s) for s in sets]


# Example usage
if __name__ == '__main__':
    sample = """
Salamence-Mega (M) @ Salamencite  
Ability: Intimidate  
Level: 50  
EVs: 2 HP / 32 Atk / 32 Spe  
Adamant Nature  
- Double-Edge  
- Roost  
- Tailwind  
- Protect  

Indeedee-F (F) @ Psychic Seed  
Ability: Psychic Surge  
Level: 50  
EVs: 32 HP / 19 Def / 15 SpD  
Calm Nature  
- Trick Room  
- Follow Me  
- Psychic  
- Hyper Voice  
"""
    
    parsed = parse_pokemon_team(sample)
    pprint.pprint(parsed)

[{'ability': 'Intimidate',
  'evs': {'atk': 32, 'hp': 2, 'spe': 32},
  'form': 'Mega',
  'gender': 'M',
  'item': 'Salamencite',
  'level': 50,
  'moves': ['Double-Edge', 'Roost', 'Tailwind', 'Protect'],
  'name': 'Salamence',
  'nature': 'Adamant',
  'shiny': False},
 {'ability': 'Psychic Surge',
  'evs': {'def': 19, 'hp': 32, 'spd': 15},
  'form': 'F',
  'gender': 'F',
  'item': 'Psychic Seed',
  'level': 50,
  'moves': ['Trick Room', 'Follow Me', 'Psychic', 'Hyper Voice'],
  'name': 'Indeedee',
  'nature': 'Calm',
  'shiny': False}]


In [ ]:
#%%manim -qm DamageFormula

class DamageFormula(Scene):
    def construct(self):
        # build first line as three pieces (start uncolored)
        left = Text("Salamence-Mega (M)", font_size=36, color=BLACK)
        at   = Text(" @ ", font_size=36, color=BLACK)
        right= Text("Salamencite", font_size=36, color=BLACK)
        first = VGroup(left, at, right).arrange(RIGHT, buff=0.06)

        # build the rest
        rest_block = Paragraph(
            "Ability: Intimidate",
            "Level: 50",
            "Shiny: Yes",
            "EVs: 5 HP / 32 Atk / 7 Def / 3 SpA / 19 Spe",
            "Naughty Nature",
            "- Double-Edge",
            "- Hyper Voice",
            "- Tailwind",
            "- Protect",
            line_spacing=0.35
        ).scale(0.7)

        # combine and add once
        group = VGroup(first, rest_block).arrange(DOWN, aligned_edge=LEFT)
        group.to_edge(LEFT, buff=1)
        self.add(group)

        # animate the rest lines falling (keep first in place)
        rest = rest_block  # Paragraph returns submobjects for each line
        down_shift = config.frame_height * DOWN
        animations = [line.animate.shift(down_shift) for line in rest]
        self.play(*animations, run_time=1.5)
        self.wait()

        # create each regex line (as you already do)
        regex_line_1 = Text("([A-Za-z0-9\\-\\s]+?)", font_size=24, color=RED)
        regex_line_2 = Text("(@)",                          font_size=24, color=YELLOW)
        regex_line_3 = Text("([A-Za-z0-9\\-\\s]+?)", font_size=24, color=GREEN)

        # stack them vertically instead of side-by-side
        regex_group = VGroup(regex_line_1, regex_line_2, regex_line_3).arrange(DOWN, buff=0.06, aligned_edge=LEFT)

        # place the stacked group under the first name part
        regex_group.next_to(first, DOWN, buff=0.15)
        regex_group.align_to(first, LEFT)

        self.play(
            Write(regex_group, run_time=0.6),
            left.animate.set_color(RED),
            at.animate.set_color(YELLOW),
            right.animate.set_color(GREEN),
        )
        self.wait()

In [7]:
mence_text = '''
Salamence-Mega @ Salamencite  
Ability: Aerilate  
Level: 50  
EVs: 2 Atk / 32 SpA / 32 Spe  
Naive Nature  
- Double-Edge  
- Hyper Voice  
- Draco Meteor  
- Flamethrower  
'''

mence_split = []
mence_split.append("Salamence-Mega @ Salamencite")
mence_split.append("Ability: Aerilate")
mence_split.append("Level: 50")
mence_split.append("EVs: 2 Atk / 32 SpA / 32 Spe")
mence_split.append("Naive Nature")
mence_split.append("- Double-Edge")
mence_split.append("- Hyper Voice")
mence_split.append("- Draco Meteor")
mence_split.append("- Flamethrower")

mence_dict = '''
{'name': 'Salamence-Mega',
'level': 50,
'evs': {'atk': 2, 'spa': 32, 'spe': 32},
'moves': ['Double-Edge',
    'Hyper Voice',
    'Draco Meteor',
    'Flamethrower'],
'item': 'Salamencite',
'ability': 'Aerilate',
'nature': 'Naive'}
'''

In [8]:
from pathlib import Path
from IPython.display import FileLink, display

videos = list(Path("media").rglob("ParseShowdown.mp4"))
print(videos)
if videos:
    display(FileLink(str(videos[-1])))

[WindowsPath('media/videos/Pokemon Stat Optimize/1080p60/ParseShowdown.mp4'), WindowsPath('media/videos/Pokemon Stat Optimize/480p15/ParseShowdown.mp4'), WindowsPath('media/videos/Pokemon Stat Optimize/720p30/ParseShowdown.mp4')]


c:\Users\evan\Desktop\Personal Projects\Pokemon Stat Optimize\media\videos\Pokemon Stat Optimize\720p30\ParseShowdown.mp4

In [9]:
%%manim -qm ParseShowdown

class ParseShowdown(Scene):
    def construct(self):
        rows = VGroup()
        move_rows = []
        move_dashes = []
        nature_rows = []
        nature_words = []
        evs_rows = []
        evs_words = []
        level_rows = []
        level_words = []
        ability_rows = []
        ability_words = []
        pokemon_rows = []
        pokemon_seperator = []
        pokemon_item = []

        for line in mence_split:
            if line.startswith("- "):
                dash = Text("- ", color=BLACK)
                move_name = Text(line[2:])
                row = VGroup(dash, move_name).arrange(RIGHT, buff=0.1)

                move_rows.append(row)
                move_dashes.append(dash)

            elif line.lower().endswith("nature"):
                nature, _ = line.split()
                nature_word = Text("Nature", color=BLACK)

                row = VGroup(
                    Text(nature),
                    nature_word,
                ).arrange(RIGHT, buff=0.2)

                nature_rows.append(row)
                nature_words.append(nature_word)

            elif line.lower().startswith("evs"):
                _, evs = line.split(" ", maxsplit=1)
                evs_word = Text("EVs:", color=BLACK)

                row = VGroup(
                    evs_word,
                    Text(evs),
                ).arrange(RIGHT, buff=0.2)

                evs_rows.append(row)
                evs_words.append(evs_word)

            elif line.lower().startswith("level"):
                _, level = line.split()
                level_word = Text("Level: ", color=BLACK)

                row = VGroup(
                    level_word,
                    Text(level),
                ).arrange(RIGHT, buff=0.2)

                level_rows.append(row)
                level_words.append(level_word)

            elif line.lower().startswith("ability"):
                _, ability = line.split()
                ability_word = Text("Ability: ", color=BLACK)

                row = VGroup(
                    ability_word,
                    Text(ability),
                ).arrange(RIGHT, buff=0.2)

                ability_rows.append(row)
                ability_words.append(ability_word)

            #elif line is first line
            elif "@" in line:
                pokemon, item = line.split("@")
                pokemon_word = Text(pokemon)
                item_word = Text(item)
                seperator = Text(" @ ", color=BLACK)

                row = VGroup(
                    pokemon_word,
                    seperator,
                    item_word,
                ).arrange(RIGHT, buff=0.08)

                pokemon_rows.append(row)
                pokemon_seperator.append(seperator)
                pokemon_item.append(item_word)

            else:
                row = Text(line)
                pokemon_rows.append(row)
                pokemon_seperator.append(Text("", color=BLACK))

            rows.add(row)

        rows.arrange(DOWN, aligned_edge=LEFT, buff=0.35)
        rows.scale(0.85)
        rows.move_to(ORIGIN)

        # Show all lines together
        self.play(FadeIn(rows))
        self.wait(0.5)

        # Highlight all four leading dashes together
        self.play(
            *[dash.animate.set_color(RED) for dash in move_dashes],
            run_time=0.5,
        )
        self.wait(2)
        self.play(
            *[FadeOut(row, shift=DOWN) for row in move_rows],
            run_time=0.8,
        )
        self.wait(0.5)

        self.play(
            *[nature.animate.set_color(RED) for nature in nature_words],
        )
        self.wait(0.5)
        self.play(
            *[FadeOut(row, shift=DOWN) for row in nature_rows],
            run_time=0.8,
        )
        self.wait(0.5)

        self.play(
            *[evs.animate.set_color(RED) for evs in evs_words],
        )
        self.wait(0.5)
        self.play(
            *[FadeOut(row, shift=DOWN) for row in evs_rows],
            run_time=0.8,
        )
        self.wait(0.5)
        
        self.play(
            *[level.animate.set_color(RED) for level in level_words],
        )
        self.wait(0.5)
        self.play(
            *[FadeOut(row, shift=DOWN) for row in level_rows],
            run_time=0.8,
        )
        self.wait(0.5)

        self.play(
            *[ability.animate.set_color(RED) for ability in ability_words],
        )
        self.wait(0.5)
        self.play(
            *[FadeOut(row, shift=DOWN) for row in ability_rows],
            run_time=0.8,
        )
        self.wait(0.5)

        self.play(
            *[seperator.animate.set_color(RED) for seperator in pokemon_seperator],
        )
        self.wait(0.5)
        self.play(
            *[FadeOut(row, shift=DOWN) for row in pokemon_rows],
            run_time=0.8,
        )
        self.wait(0.5)

Manim Community v0.20.1

In [10]:
%%manim -qm showdict

class showdict(Scene):
    def construct(self):
        #fade in mence_dict
        mence_dict_text = Text(mence_dict, font_size=36)
        self.play(FadeIn(mence_dict_text, shift=UP))
        self.wait(3)


Manim Community v0.20.1

In [ ]:
damage_calced = '''[{'attacker': 'Salamence-Mega',
  'defender': 'Delphox-Mega',
  'move': 'Double-Edge',
  'category': 'Physical',
  'damage': [148,
   150,
   151,
   153,
   156,
   157,
   159,
   160,
   162,
   163,
   166,
   168,
   169,
   171,
   172,
   175],
  'maxDamage': 175,
  'minDamage': 148,
  'description': '2 Atk Aerilate Salamence-Mega Double-Edge vs. 2 HP / 0 Def Delphox-Mega: 148-175 (97.3 - 115.1%) -- 81.3% chance to OHKO',
  'attackerStats': {'hp': 170,
   'atk': 167,
   'def': 150,
   'spa': 172,
   'spd': 99,
   'spe': 189},
  'defenderStats': {'hp': 152,
   'atk': 80,
   'def': 92,
   'spa': 211,
   'spd': 145,
   'spe': 204}},'''

attacker = ''''attacker': {'name': 'Salamence-Mega',
   'level': 50,
   'evs': {'atk': 2, 'spa': 32, 'spe': 32},
   'item': 'Salamencite',
   'ability': 'Aerilate',
   'nature': 'Naive'},'''

defender = ''''defender': {'name': 'Delphox-Mega',
   'level': 50,
   'evs': {'hp': 2, 'spa': 32, 'spe': 32},
   'item': 'Delphoxite',
   'ability': 'Levitate',
   'nature': 'Timid'},'''

atk_field = '''
  'move': 'Double-Edge',
  'field': {'gameType': 'Doubles'}},'''

: 